# 01 - Data cleaning and quality
Hotel Booking Demand (Antonio, Almeida & Nunes 2019, via TidyTuesday): 119,390 bookings from a Lisbon City Hotel and an Algarve Resort Hotel, arrivals Jul 2015 - Aug 2017.

Goal of this notebook: a clean fact table a BI tool can trust, with every removed row accounted for.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.width', 140)

In [2]:
raw = pd.read_csv('../data/raw/hotel_bookings.csv', na_values=['NULL'])
print(raw.shape)
raw.head(3)

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


## Shape and first checks

In [3]:
print('columns:', len(raw.columns))
print(raw.dtypes.value_counts())
raw.isna().sum()[raw.isna().sum() > 0]

columns: 32
int64      16
str        12
float64     4
Name: count, dtype: int64


children         4
country        488
agent        16340
company     112593
dtype: int64

Missing values arrive as the literal string `NULL` in the source file, which is why `na_values=['NULL']` matters: without it, `country` looks complete and `agent`/`company` look like text.

In [4]:
print('full duplicate rows:', raw.duplicated().sum())

full duplicate rows: 31994


**31,994 byte-identical rows - and we keep every one.** The source has no booking ID, so two identical rows are two separate bookings that happen to share every attribute (same hotel, dates, room, price). Dropping them would silently delete about 27% of all bookings. No dedupe in this project.

In [5]:
MONTH = {m: i+1 for i, m in enumerate(['January','February','March','April','May','June','July','August','September','October','November','December'])}
df = raw.copy()
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['arrival_date'] = pd.to_datetime({'year': df['arrival_date_year'], 'month': df['arrival_date_month'].map(MONTH), 'day': df['arrival_date_day_of_month']})
df['booking_date'] = df['arrival_date'] - pd.to_timedelta(df['lead_time'], unit='D')
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['guests'] = df['adults'] + df['children'] + df['babies']
print(df['arrival_date'].min(), '->', df['arrival_date'].max())

2015-07-01 00:00:00 -> 2017-08-31 00:00:00


In [6]:
print('zero-night rows:', (df['total_nights'] == 0).sum())
print('adr == 0 rows:', (df['adr'] == 0).sum())
print('adr max:', df['adr'].max())
print('no-guest rows:', (df['guests'] == 0).sum())

zero-night rows: 715
adr == 0 rows: 1959
adr max: 5400.0
no-guest rows: 180


Findings: 715 zero-night rows and 1,960 zero-ADR rows are mostly complimentary / day-use bookings - kept and flagged, revenue is honestly 0. The single ADR of 5,400 is an outlier that would poison every average it touches.

In [7]:
print(df.groupby('reservation_status').size())
print('No-Show rows where source is_canceled = 1:', ((df['reservation_status'] == 'No-Show') & (df['is_canceled'] == 1)).sum())

reservation_status
Canceled     43017
Check-Out    75166
No-Show       1207
dtype: int64
No-Show rows where source is_canceled = 1: 1207


**Source quirk:** all 1,207 No-Show rows carry `is_canceled = 1` in the source. Cancellation rate in this project therefore means canceled + no-show, and realized revenue is defined separately - only `Check-Out` earns revenue.

In [8]:
quarantine = (df['guests'] == 0) | (df['adr'] > 1000)
print('quarantined:', quarantine.sum())
clean = df[~quarantine].copy()
quar = df[quarantine].copy()
assert len(raw) == len(clean) + len(quar), 'reconciliation failed'
print(f'reconciliation OK: {len(raw)} = {len(clean)} clean + {len(quar)} quarantined')

quarantined: 181


reconciliation OK: 119390 = 119209 clean + 181 quarantined


Quarantine rule is deliberately narrow: a row leaves the clean fact only if it cannot describe a real stay (no guests at all, or the 5,400 ADR outlier). Everything else stays, flagged.

In [9]:
costs = pd.read_csv('../data/reference/channel_costs.csv')
print(costs)
clean = clean.merge(costs, on='distribution_channel', how='left', validate='many_to_one')

  distribution_channel  commission_rate  acquisition_cost_per_booking_eur  is_synthetic
0               Direct             0.03                               6.0          True
1            Corporate             0.08                               4.0          True
2                  GDS             0.12                              15.0          True
3                TA/TO             0.18                              22.0          True
4            Undefined             0.10                              10.0          True


Channel costs are a **synthetic reference layer, labelled as such** (`is_synthetic = TRUE`): the source has no commission or acquisition cost, and margin / ROI cannot be computed without one. Rates are stated in the file, never hidden in code.

In [10]:
clean['booked_revenue_eur'] = clean['adr'] * clean['total_nights']
clean['is_realized'] = (clean['reservation_status'] == 'Check-Out').astype(int)
clean['is_noshow'] = (clean['reservation_status'] == 'No-Show').astype(int)
clean['realized_revenue_eur'] = clean['booked_revenue_eur'] * clean['is_realized']
clean['commission_cost_eur'] = clean['realized_revenue_eur'] * clean['commission_rate']
clean['acquisition_cost_eur'] = clean['acquisition_cost_per_booking_eur'] * clean['is_realized']
clean['channel_cost_eur'] = clean['commission_cost_eur'] + clean['acquisition_cost_eur']
clean['gross_margin_eur'] = clean['realized_revenue_eur'] - clean['channel_cost_eur']
clean['arrival_week'] = clean['arrival_date'].dt.to_period('W-SUN').apply(lambda p: p.start_time)
clean['arrival_month'] = clean['arrival_date'].dt.to_period('M').dt.to_timestamp()
print(f"realized revenue EUR {clean['realized_revenue_eur'].sum():,.0f} | gross margin EUR {clean['gross_margin_eur'].sum():,.0f}")

realized revenue EUR 25,986,912 | gross margin EUR 20,679,670


In [11]:
clean.to_csv('../data/fact_bookings_clean.csv', index=False)
quar.to_csv('../data/quarantined_rows.csv', index=False)
print('saved fact + quarantine')

saved fact + quarantine


## Summary
- Raw 119,390 -> clean 119,209 + 181 quarantined, reconciled exactly.
- Kept on purpose: 31,994 identical-looking rows (no booking ID), 645 zero-night rows, 1,810 zero-ADR complimentary rows.
- Flagged, not hidden: No-Show counted as canceled by the source; synthetic channel-cost layer.